In [66]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [67]:
df=pd.read_csv('Churn_Modelling.csv')

In [68]:
df

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [69]:
df.drop(columns=['RowNumber', 'CustomerId','Surname'], inplace=True)

In [63]:
#puting it latter for geting pkl

from sklearn.preprocessing import LabelEncoder
import pickle

gender_encoder = LabelEncoder()

df["Gender"] = gender_encoder.fit_transform(df["Gender"])

with open("gender_encoder.pkl", "wb") as f:
    pickle.dump(gender_encoder, f)

In [70]:
df.Gender=df['Gender'].map({'Female':0,'Male':1})

In [64]:
## i added it latter for geting encoding of one hot encoder for geography column

from sklearn.preprocessing import OneHotEncoder
import pickle

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[['Geography']])

with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

In [72]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1,0,0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1,0,0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1,0,0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0,1,0


In [71]:
df = pd.get_dummies(df, columns=['Geography'], dtype=int)

In [73]:
import pickle

In [9]:
with open("feature_columns.pkl", "wb") as file:
    pickle.dump(df.columns.tolist(), file)

In [74]:
X = df.drop('Exited', axis=1)
y = df['Exited']

In [75]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y      # Recommended for classification
)

In [76]:
from sklearn.preprocessing import StandardScaler

num_cols = ['CreditScore', 'Age', 'Balance', 'Tenure', 'NumOfProducts', 'EstimatedSalary']

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [77]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [81]:
X_train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
2151,1.058568,1,1.715086,0.684723,-1.226059,-0.910256,1,0,1.042084,1,0,0
8392,0.913626,1,-0.659935,-0.696202,0.413288,-0.910256,1,0,-0.623556,0,1,0
5006,1.079274,0,-0.184931,-1.731895,0.601687,0.808830,1,1,0.308128,0,1,0
4117,-0.929207,1,-0.184931,-0.005739,-1.226059,0.808830,1,0,-0.290199,1,0,0
7182,0.427035,1,0.955079,0.339492,0.548318,0.808830,0,1,0.135042,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4555,0.385623,0,-0.374933,0.339492,-1.226059,-0.910256,1,0,-1.294966,0,0,1
4644,0.634095,1,3.330101,-0.005739,-1.226059,0.808830,0,0,0.901685,0,0,1
8942,0.168210,0,-0.184931,1.375185,-0.073747,0.808830,1,1,-0.558088,1,0,0
2935,0.375270,1,-0.374933,1.029954,0.394991,0.808830,1,0,-1.351500,0,0,1


## ANN implementation

In [82]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [83]:
## build ann model
model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), #hl1
    Dense(32, activation='relu'),#hl2
    Dense(1, activation='sigmoid')#output layer
])


c:\Users\ankit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [84]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [85]:
opt=tf.keras.optimizers.Adam(learning_rate=0.001)


In [86]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['recall'])

In [87]:
## set up tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [88]:
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [89]:
# setup early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [90]:
# train the model
history =model.fit(
    X_train,
    y_train,validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorflow_callback]


)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4392 - recall: 0.1454 - val_loss: 0.3928 - val_recall: 0.2850
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3707 - recall: 0.3828 - val_loss: 0.3577 - val_recall: 0.3391
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3524 - recall: 0.4337 - val_loss: 0.3425 - val_recall: 0.4840
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3439 - recall: 0.4632 - val_loss: 0.3403 - val_recall: 0.4840
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3395 - recall: 0.4669 - val_loss: 0.3457 - val_recall: 0.4324
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3357 - recall: 0.4656 - val_loss: 0.3373 - val_recall: 0.4889
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3326 - recall: 0.4742 - val_loss: 0.3378 - val_recall: 0.4717
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3312 - recall: 0.4785 - val_loss: 0.3382 - val_recall: 0.3931


In [91]:
model.save('model3.h5')

In [47]:
model.save('model.h5')

In [48]:
# load tensor board extenstion
%load_ext tensorboard

In [50]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 25432), started 0:01:16 ago. (Use '!kill 25432' to kill it.)